In [ ]:
# ==========================================================
# Improved RAG Pipeline with Hybrid Search (Vector + Keyword)
# ==========================================================

from neo4j import GraphDatabase
from langchain_huggingface import HuggingFaceEmbeddings
from PyPDF2 import PdfReader
from openai import OpenAI

NEO4J_URI = ""
NEO4J_USER = ""
NEO4J_PASSWORD = ""

# open_ai_client = OpenAI(api_key=OPENAI_API_KEY)
hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD)
)

In [27]:
# ==========================================================
# PDF READER
# ==========================================================

def read_pdf(file_path):

    reader = PdfReader(file_path)
    text = ""

    for page in reader.pages:
        text += page.extract_text() or ""

    return text


In [28]:
# ==========================================================
# TEXT CHUNKING
# ==========================================================

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk.strip())

        start += chunk_size - overlap

    return chunks

In [29]:
# ==========================================================
# EMBEDDING FUNCTION
# ==========================================================

def embed(texts):

    return hf_embeddings.embed_documents(texts)

In [30]:
# ==========================================================
# CREATE VECTOR INDEX
# ==========================================================

def create_vector_index():

    query = """
    CREATE VECTOR INDEX pdf IF NOT EXISTS
    FOR (c:Chunk)
    ON c.embedding
    """

    driver.execute_query(query)

In [31]:
# ==========================================================
# CREATE FULLTEXT INDEX
# ==========================================================

def create_fulltext_index():

    query = """
    CREATE FULLTEXT INDEX ftPdfChunk IF NOT EXISTS
    FOR (c:Chunk)
    ON EACH [c.text]
    """

    driver.execute_query(query)

In [32]:
# ==========================================================
# STORE CHUNKS IN NEO4J
# ==========================================================

def store_chunks(chunks, embeddings):

    cypher_query = """
    WITH $chunks as chunks, range(0, size($chunks)-1) AS index
    UNWIND index AS i
    WITH i, chunks[i] AS chunk, $embeddings[i] AS embedding
    MERGE (c:Chunk {index: i})
    SET c.text = chunk,
        c.embedding = embedding
    """

    driver.execute_query(
        cypher_query,
        chunks=chunks,
        embeddings=embeddings
    )

In [33]:
# ==========================================================
# HYBRID SEARCH (VECTOR + FULLTEXT)
# ==========================================================

def hybrid_search(question, question_embedding, k=4):

    hybrid_query = """
    CALL {

        CALL db.index.vector.queryNodes('pdf', $k, $question_embedding)
        YIELD node, score
        WITH collect({node:node, score:score}) AS nodes, max(score) AS max
        UNWIND nodes AS n
        RETURN n.node AS node, (n.score / max) AS score

        UNION

        CALL db.index.fulltext.queryNodes('ftPdfChunk', $question, {limit:$k})
        YIELD node, score
        WITH collect({node:node, score:score}) AS nodes, max(score) AS max
        UNWIND nodes AS n
        RETURN n.node AS node, (n.score / max) AS score
    }

    WITH node, max(score) AS score
    ORDER BY score DESC
    LIMIT $k

    RETURN node, score
    """

    records, _, _ = driver.execute_query(
        hybrid_query,
        question_embedding=question_embedding,
        question=question,
        k=k
    )

    return records

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(r".env", override=True)

hf_token = os.getenv("GROQ_API_KEY")

In [35]:
from groq import Groq

client = Groq(api_key=hf_token)

In [36]:
# ==========================================================
# GENERATE ANSWER USING LLM
# ==========================================================

def generate_answer(question, records):

    context = "\n".join([doc["node"]["text"] for doc in records])

    prompt = f"""
You are a precise and reliable assistant.

Answer the question ONLY using the information provided in the context.

Rules:
- Do not use external knowledge
- If context does not contain answer say: "I don't know"

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=500
    )

    answer = response.choices[0].message.content

    print("\n==============================")
    print("QUESTION:")
    print(question)

    print("\nANSWER:")
    print(answer)
    print("==============================")

In [37]:
# ==========================================================
# MAIN PIPELINE
# ==========================================================

def main():

    print("Reading PDF...")

    text = read_pdf("CH1_SuraAlFatiha_2Edition.pdf")

    print("Chunking text...")

    chunks = chunk_text(text)

    print("Total Chunks:", len(chunks))

    print("Generating embeddings...")

    embeddings = embed(chunks)

    print("Embedding dimension:", len(embeddings[0]))

    print("Creating indexes...")

    create_vector_index()
    create_fulltext_index()

    print("Storing chunks in Neo4j...")

    store_chunks(chunks, embeddings)

    # Ask Question
    question = "Who is most merciful?"

    print("\nCreating question embedding...")

    question_embedding = embed([question])[0]

    print("\nRunning Hybrid Search...")

    records = hybrid_search(question, question_embedding)

    print("\nRetrieved Chunks:\n")

    for r in records:

        print(r["node"]["text"])
        print("Score:", r["score"])
        print("Index:", r["node"]["index"])
        print("------")

    generate_answer(question, records)


# ==========================================================
# RUN
# ==========================================================

if __name__ == "__main__":
    main()

Reading PDF...
Chunking text...
Total Chunks: 274
Generating embeddings...
Embedding dimension: 768
Creating indexes...
Storing chunks in Neo4j...

Creating question embedding...

Running Hybrid Search...


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=2, column=5, offset=5>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 5, 'line': 2, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    CALL {\n\n        CALL db.index.vector.queryNodes('pdf', $k, $question_embedding)\n        YIELD node, score\n        WITH collect({node:node, score:score}) AS nodes, max(score) AS max\n        UNWIND nodes AS n\n        RETURN n.node AS node, (n.score / max) AS score\n\n        UNION\n\n        CALL db.index.fulltext.queryNodes('ftPdfChun


Retrieved Chunks:

Allah the Lord of the worlds  [1:2] : ‘The Beneficent  to all of His-azwj creation, The Merciful  
to the Believers in particular.  Master of the Day of Reckoning  ‘[1:4]  - Day of Reckoni ng’.53 
 .وروى عن الصادق عليه السالم انه قال: الرْحن اسم خاص بصفة عامة والرحيم اسم عام بصفة خاصة 
And it has been narrated from Al-Sadiq-asws: ‘The Beneficent  is a speci al Name with gener al 
Characteristics, and the Merciful  is a gener al Name with speci al Characteristics ’.54 
 .يف هنج البالغة: رحيم ال
Score: 1.0
Index: 164
------
ه السعالم) يقول- قاتلهم هللا- عمدوا إىل أعظم آية يف كتاب هللا، فزعموا أهنا بدعة إذا 
.»أظهروها، و هي بِسْمِ ارَِّ الرَّْحْنِ الرَّحِيمِ   
From Khalid Bin Al Mukhtar who said,  
‘I heard Ja’far-asws Bin Muhammad-asws saying: ‘What is the matter with them? May Allah-azwj 
Fight  them! They are deliberating to the most magnificent Verse in the Book of Allah-azwj, and 
they are claiming that it is an innovation when they are manifesting it, and it is 